# Install Depndancy

In [ ]:
! pip install -q json-repair  qwen-vl-utils python-docx bitsandbytes hf_transfer

In [ ]:
!pip install vllm==0.19.1

# VLM

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Load Data

In [ ]:
import os
DRIVE_BASE    = '/content/drive/MyDrive/Iran Israel War/extracted_map_layers'    # your project folder
# INPUT_CSV     = os.path.join(DRIVE_BASE, 'final_extracted_events.csv')
INPUT_CSV     = os.path.join(DRIVE_BASE, 'master_unified_campaign_log.csv')
OUTPUT_CSV    = os.path.join(DRIVE_BASE, 'bda_assessed_final_report.csv')
OUTPUT_DOCX   = os.path.join(DRIVE_BASE, 'BDA_Final_Dossier.docx')


os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {VLM_MODEL_ID}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War/extracted_map_layers


In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)

In [ ]:
import pandas as pd

df_raw = pd.read_csv(INPUT_CSV)

# Keep only rows that have both image paths
valid_analysis_df = df_raw.copy().reset_index(drop=True)

# print(f'✅ Total rows     : {len(df_raw)}')
# print(f'   Processable   : {len(valid_analysis_df)}')
# print(f'   Skipped       : {len(df_raw) - len(valid_analysis_df)} (missing image paths)')
# valid_analysis_df[['event_full_title', 'image_path_sat', 'image_path_map']].head()

## Load VLM

In [ ]:
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from docx import Document
from docx.shared import Inches
from transformers import AutoProcessor
import torch
# ==============================================================================
# PHASE 2: RIGID VLM ASSESSMENT (TWO IMAGES)
# ==============================================================================
print("\n--- PHASE 2: NATIVE MULTIMODAL BDA ---")

num_gpus = torch.cuda.device_count()
print(f"[INFO] Detected {num_gpus} GPUs. Splitting model across them...")




--- PHASE 2: NATIVE MULTIMODAL BDA ---
[INFO] Detected 1 GPUs. Splitting model across them...


## Build and Run Inference

In [ ]:

# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 8192  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War/extracted_map_layers
   N samples  : 5
   Temperature: 0.3
   Max tokens : 8192


In [ ]:
import os

DEST = "/content/images"

if not os.path.exists(DEST):
    os.makedirs(DEST)

    # Extract both zips
    !unzip -q "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/crater_visualisations_part_1.zip" -d /content/temp_extract
    !unzip -q "/content/drive/MyDrive/Iran Israel War/extracted_map_layers/crater_visualisations_part_2.zip" -d /content/temp_extract


    # Move files into images
    !mv /content/temp_extract/crater_visualisations/* /content/images/

    # Cleanup
    !rm -rf /content/temp_extract

In [ ]:

# !rm -rf  /content/images/

In [ ]:
import os


num_files = sum(len(files) for _, _, files in os.walk("/content/images"))
print(f"Total files: {num_files}")

Total files: 6360


In [ ]:
# from google.colab import auth
# from googleapiclient.discovery import build

# auth.authenticate_user()
# service = build('drive', 'v3')

# restored = 0
# page_token = None

# while True:
#     response = service.files().list(
#         q="trashed = true",
#         spaces='drive',
#         fields='nextPageToken, files(id, name, parents)',
#         pageToken=page_token
#     ).execute()

#     files = response.get('files', [])

#     for f in files:
#         service.files().update(
#             fileId=f['id'],
#             body={'trashed': False}
#         ).execute()
#         print(f"✅ Restored: {f['name']}")
#         restored += 1

#     page_token = response.get('nextPageToken', None)
#     if not page_token:
#         break

# print(f"\n🎉 Total restored: {restored}")

In [ ]:
# ! rm -r /content/images

In [ ]:
# valid_analysis_df=valid_analysis_df.iloc[1:]

In [ ]:
vlm_prompt = """\
You are a Geospatial Intelligence Analyst counting buildings impacted by a disaster event.

IMAGES PROVIDED:
  Image 1 — Side-by-side satellite comparison (Google LEFT, ESRI RIGHT):
    • Cyan dashed circle = impact zone boundary
    • Green outlines  = building detected by both sources (complete match)
    • Yellow outlines = building partially matched between sources
    • Magenta outlines = building detected by one source only — do NOT miss these
    • Grey outlines = outside zone of interest, ignore

    ⚠️ LABEL WARNING: Polygon labels (P-1, P-2, C-1, etc.) are assigned INDEPENDENTLY
    per source. P-1 in Google and P-1 in ESRI are NOT necessarily the same building.
    Do NOT match buildings by label — match them by spatial position and shape only.

    ⚠️ OUTLINE WARNING: Outlines are algorithmic detections — they are NOT ground truth.
    Both sources can simultaneously miss a real building, leaving it completely unoutlined.
    You MUST visually inspect the raw imagery inside the cyan circle on BOTH panels
    for any rooftop structure that has no outline at all. If you see a clear rooftop
    with no polygon over it, count it — do not rely solely on outlines.

    True count = UNION of:
      (a) All outlined polygons from both sources (deduplicated by location)
      (b) Any additional unoutlined rooftops you visually confirm in the raw imagery

  Image 2 — Per-target zoomed 4-panel grid (one row per detected target):
    • Col 1 (yellow outline): Google RGB — zoomed crop of the target
    • Col 2 (cyan outline):   ESRI RGB  — zoomed crop of the same region
    • Col 3: Google RELATIVE depth map
    • Col 4: ESRI RELATIVE depth map

    ⚠️ DEPTH WARNING: These are RELATIVE depth maps — colors encode elevation
    relative to other pixels in that same crop, NOT absolute height in metres.
    Use depth to:
      • Confirm a blob is a building (relatively elevated vs surrounding ground)
      • Separate merged blobs — two distinct elevation plateaus = two buildings
      • Find buildings missed by BOTH sources — a clear elevated region in depth
        with NO outline in either RGB panel is a strong candidate for a missed building.
        Cross-check against both RGB panels — if a rooftop is visible there too, count it.

WHAT TO COUNT:
  Count every intact building roof that intersects or lies inside the cyan dashed circle.
  Touching the boundary = inside. Skip trees, vehicles, shadows, ruins.

VERDICT DEFINITIONS:
  Fully Inside     : entire roof within the cyan circle
  Partially Inside : cyan boundary cuts through the roof
  Outside          : roof entirely outside — do not count

Output this block EXACTLY:
FULLY_INSIDE: {integer}
PARTIALLY_INSIDE: {integer}
TOTAL_IMPACTED: {integer}
==================END==================
"""
KICKSTART = "<think>\n"

# ── SAMPLING CONFIG ───────────────────────────────────────────────────────────
N_SAMPLES   = 5      # >1 required for median consensus to be meaningful
TEMPERATURE = 0.3    # low but non-zero so samples can differ
MAX_TOKENS  = 8192  # free reasoning needs room
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'✅ Drive mounted. Project folder: {DRIVE_BASE}')
# print(f'   Model      : {model_id}')
print(f'   N samples  : {N_SAMPLES}')
print(f'   Temperature: {TEMPERATURE}')
print(f'   Max tokens : {MAX_TOKENS}')

STOP = ["==================END=================="]
import os
import gc
import time
import torch
import shutil
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from huggingface_hub import snapshot_download

# IMPORT THE INTERNAL vLLM CLEANUP FUNCTION
from vllm.distributed.parallel_state import destroy_model_parallel


# =========================
# HuggingFace cache (fast + persistent)
# =========================
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# =========================
# MinerU high-performance mode
# =========================
os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

# =========================
# vLLM stability + performance
# =========================
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"
# os.environ["VLLM_ALLOW_LONG_MAX_MODEL_LEN"] = "8192"

# =========================
# CUDA stability fixes
# =========================
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

print("✅ Environment ready (vLLM performance mode)")

# ==============================================================================
# 2. DEFINE MODELS TO RUN
# ==============================================================================

SHORTER_LEN_MODELS=['deepseek-ai/deepseek-vl2','llava-hf/llava-v1.6-34b-hf',]

MODELS_TO_RUN = [
    #DONE
    "Qwen/Qwen3.6-35B-A3B",
    'google/gemma-4-31B-it',
     'zai-org/GLM-4.6V-Flash',
    'sakamakismile/Huihui-Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-NVFP4',
     'nvidia/Cosmos-Reason2-32B',
    #######

]
HIGH_VISION_TOKEN_MODELS = [
    "sakamakismile/Huihui-Qwen3.6-35B-A3B-Claude-4.7-Opus-abliterated-NVFP4",
]
DRIVE_BASE = "/content/drive/MyDrive/new2/Outputs_3" # Adjust to your actual path
os.makedirs(DRIVE_BASE, exist_ok=True)

# ==============================================================================
# 3. MAIN EXECUTION LOOP
# ==============================================================================
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

def load_and_resize(path, max_width=1600):
    img = Image.open(path).convert('RGB')
    if img.width > max_width:
        ratio = max_width / img.width
        new_size = (max_width, int(img.height * ratio))
        img = img.resize(new_size, Image.LANCZOS)
    return img
def load_and_resize(path, max_pixels=None):
    img = Image.open(path).convert('RGB')
    if max_pixels is not None:
        w, h = img.size
        if w * h > max_pixels:
            scale = (max_pixels / (w * h)) ** 0.5
            img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

for model_id in MODELS_TO_RUN:
    print(f"\n{'='*60}")
    print(f"🚀 STARTING RUN FOR MODEL: {model_id}")
    print(f"{'='*60}")

    # --- Step A: Download Model ---
    print(f"\n[INFO] Downloading {model_id} to Disk...")
    snapshot_download(repo_id=model_id, resume_download=True, max_workers=8)
    print("[INFO] Download complete! Model is now cached on disk.")

    # --- Step B: Load Processor & VLM ---
    print(f'[INFO] Loading processor: {model_id}')
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    Max_len = 4096 if model_id in SHORTER_LEN_MODELS else 32768

    print(f'[INFO] Loading vLLM engine: {model_id}')
    vlm_llm = LLM(
        model=model_id,
        max_model_len=Max_len,
        trust_remote_code=True,
        limit_mm_per_prompt={"image": 2},
        gpu_memory_utilization=0.90,
        enforce_eager=True,
        disable_log_stats=False,
    )
    print('✅ VLM loaded successfully')

    # --- Step C: Build Prompt Inputs ---
    vlm_inputs = []    # ✅ reset per model
    missing_rows = []  # ✅ reset per model
    print(f'[INFO] Building inputs for {len(valid_analysis_df)} rows...')
    try:
      valid_analysis_df.rename(columns={'latitude': 'lat_dec','longitude':'lon_dec'}, inplace=True)
    except:
      pass
    for idx, row in tqdm(valid_analysis_df.iterrows(), total=len(valid_analysis_df)):

        path_highlighted = os.path.join('/content/images', f"row_{idx}_comparison_outlined_highlighted.png")
        path_depth       = os.path.join('/content/images', f"row_{idx}_grid_4panel_depth.jpg")

        if not os.path.exists(path_highlighted) or not os.path.exists(path_depth):
            missing_rows.append(idx)
            continue

        messages = [
            {
                'role': 'system',
                'content': (
                    'You are a strict, objective imagery analyst. '
                    'Only count clear, distinct, intact physical buildings. '
                    'Do not guess or infer structures that are not clearly visible.'
                ),
            },
            {
                'role': 'user',
                'content': [
                    {'type': 'image'},
                    {'type': 'image'},
                    {
                        'type': 'text',
                        'text': (
                            f"Event Context: {row['event_full_title']}\n"
                            f"Location: {row['lat_dec']}, {row['lon_dec']}\n\n"
                            f"{vlm_prompt}"
                        ),
                    },
                ],
            },
        ]

        # img_outline_highlighted = load_and_resize(path_highlighted, max_width=1920)
        # img_4_panel_with_depth  = load_and_resize(path_depth,       max_width=1280)
        max_pixels = 500_000 if model_id in HIGH_VISION_TOKEN_MODELS else None  # None = no resize limit

        img_outline_highlighted = load_and_resize(path_highlighted, max_pixels=max_pixels)
        img_4_panel_with_depth  = load_and_resize(path_depth,       max_pixels=max_pixels)
        prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        prompt += KICKSTART

        vlm_inputs.append({
            'prompt': prompt,
            'multi_modal_data': {'image': [img_outline_highlighted, img_4_panel_with_depth]},
            '_row_idx': idx,
        })

    print(f'✅ Built {len(vlm_inputs)} inputs ({len(missing_rows)} rows skipped — images missing)')

    # --- Step D: Inference ---
    vllm_safe_inputs = [
        {k: v for k, v in inp.items() if k != '_row_idx'}
        for inp in vlm_inputs
    ]
    row_indices = [inp['_row_idx'] for inp in vlm_inputs]

    print(f'[INFO] Running VLM inference (n={N_SAMPLES}, temp={TEMPERATURE}, max_tokens={MAX_TOKENS})...')
    vlm_outputs = vlm_llm.generate(
        vllm_safe_inputs,
        SamplingParams(n=N_SAMPLES, temperature=TEMPERATURE, max_tokens=MAX_TOKENS),
        use_tqdm=True,
    )
    print(f'✅ Inference complete — {len(vlm_outputs)} results')

    # --- Step E: Extract Data ---
    data_for_df = []  # ✅ reset per model
    output_lookup = {row_indices[i]: vlm_outputs[i] for i in range(len(vlm_outputs))}

    for idx, row in valid_analysis_df.iterrows():
        row_data = row.to_dict()

        if idx in missing_rows:
            for j in range(N_SAMPLES):
                row_data[f'output_sample_{j+1}'] = (
                    "FULLY_INSIDE: 0\nPARTIALLY_INSIDE: 0\nTOTAL_IMPACTED: 0\n"
                    "==================END==================\n[SKIPPED: missing images]"
                )
        elif idx in output_lookup:
            outputs = output_lookup[idx].outputs
            for j, output in enumerate(outputs):
                row_data[f'output_sample_{j+1}'] = output.text

        data_for_df.append(row_data)
    # ✅ END of row loop — CSV save is now correctly outside it

    # --- Save CSV ---
    vlm_outputs_df = pd.DataFrame(data_for_df)
    safe_model_name = model_id.replace("/", "_").replace("-", "_")
    output_vlm_raw_csv = os.path.join(DRIVE_BASE, f'vlm_raw_outputs_{safe_model_name}.csv')
    vlm_outputs_df.to_csv(output_vlm_raw_csv, index=False)
    print(f"✅ Raw VLM outputs saved to: {output_vlm_raw_csv}")

    # --- Step F: Memory Cleanup ---
    print(f"[INFO] Unloading {model_id} and tearing down vLLM state...")
    del vlm_llm
    del processor
    destroy_model_parallel()

    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

    gc.collect()
    torch.cuda.empty_cache()
    print(f"[INFO] VRAM cleared.")

    # --- Step G: Disk Cleanup ---
    print(f"[INFO] Deleting model files from disk to free up storage...")
    hf_folder_name = f"models--{model_id.replace('/', '--')}"
    model_cache_path = os.path.join(os.environ["HF_HOME"], "hub", hf_folder_name)

    if os.path.exists(model_cache_path):
        shutil.rmtree(model_cache_path)
        print(f"✅ Deleted disk cache for {model_id}: {model_cache_path}")
    else:
        print(f"⚠️ Cache directory not found, skipped: {model_cache_path}")

    print(f"[INFO] Ready for next model.\n")

print("\n🎉 ALL MODELS PROCESSED AND CLEARED SUCCESSFULLY!")

✅ Drive mounted. Project folder: /content/drive/MyDrive/Iran Israel War/extracted_map_layers
   N samples  : 5
   Temperature: 0.3
   Max tokens : 8192
✅ Environment ready (vLLM performance mode)

🚀 STARTING RUN FOR MODEL: google/gemma-4-31B-it

[INFO] Downloading google/gemma-4-31B-it to Disk...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

[INFO] Download complete! Model is now cached on disk.
[INFO] Loading processor: google/gemma-4-31B-it
[INFO] Loading vLLM engine: google/gemma-4-31B-it
INFO 06-27 02:42:15 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 32768, 'enforce_eager': True, 'limit_mm_per_prompt': {'image': 2}, 'model': 'google/gemma-4-31B-it'}
WARNING 06-27 02:42:15 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 06-27 02:42:28 [model.py:549] Resolved architecture: Gemma4ForConditionalGeneration
INFO 06-27 02:42:28 [model.py:1678] Using max model len 32768
INFO 06-27 02:42:28 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 06-27 02:42:28 [config.py:104] Gemma4 model has heterogeneous head dimensions (head_dim=256, global_head_dim=512). Forcing TRITON_ATTN backend to prevent mixed-backend numerical divergence.
INFO 06-27 02:42:28 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 06-27 02:42:28 [vllm.py:848] 

  0%|          | 0/893 [00:00<?, ?it/s]

✅ Built 610 inputs (283 rows skipped — images missing)
[INFO] Running VLM inference (n=5, temp=0.3, max_tokens=8192)...


Rendering prompts:   0%|          | 0/610 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/3050 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

INFO 06-27 02:48:33 [loggers.py:259] Engine 000: Avg prompt throughput: 395.3 tokens/s, Avg generation throughput: 244.7 tokens/s, Running: 67 reqs, Waiting: 2855 reqs, GPU KV cache usage: 93.3%, Prefix cache hit rate: 41.3%, MM cache hit rate: 4.4%
INFO 06-27 02:48:43 [loggers.py:259] Engine 000: Avg prompt throughput: 640.2 tokens/s, Avg generation throughput: 646.6 tokens/s, Running: 75 reqs, Waiting: 2825 reqs, GPU KV cache usage: 98.5%, Prefix cache hit rate: 38.3%, MM cache hit rate: 4.4%
INFO 06-27 02:48:53 [loggers.py:259] Engine 000: Avg prompt throughput: 940.7 tokens/s, Avg generation throughput: 603.4 tokens/s, Running: 69 reqs, Waiting: 2790 reqs, GPU KV cache usage: 97.2%, Prefix cache hit rate: 35.9%, MM cache hit rate: 4.4%
INFO 06-27 02:49:03 [loggers.py:259] Engine 000: Avg prompt throughput: 775.8 tokens/s, Avg generation throughput: 635.4 tokens/s, Running: 75 reqs, Waiting: 2760 reqs, GPU KV cache usage: 96.4%, Prefix cache hit rate: 35.0%, MM cache hit rate: 4.4%


## Nu Extract

In [ ]:
# ==============================================================================
# NuExtract-2.0-8B EXTRACTION PIPELINE
# ==============================================================================

import os
import gc
import re
import json
import torch
import shutil
import statistics
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from vllm import LLM, SamplingParams
from vllm.distributed.parallel_state import destroy_model_parallel
from huggingface_hub import snapshot_download


# ==============================================================================
# 1. ENVIRONMENT CONFIGURATION
# ==============================================================================

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

os.environ["MINERU_BACKEND"] = "vllm"
os.environ["MINERU_VLLM_ENABLE"] = "true"

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"


# ==============================================================================
# SECTION 1 — CONFIG
# ==============================================================================
NUEXTRACT_MODEL_ID = "numind/NuExtract-2.0-8B"
DRIVE_BASE         = "/content/drive/MyDrive/new/Outputs_3"
HF_HOME            = os.environ.get("HF_HOME", "/content/hf_cache")
N_SAMPLES          = 5

NUEXTRACT_TEMPLATE = json.dumps(
    {
        "Buildings_Fully_Inside":     0,
        "Buildings_Partially_Inside": 0,
        "Total_Buildings_Impacted":   0,
    },
    indent=2,
    ensure_ascii=False,
)

print(f"   Model     : {NUEXTRACT_MODEL_ID}")
print(f"   DRIVE_BASE: {DRIVE_BASE}")
print(f"   N_SAMPLES : {N_SAMPLES}")
print(f"   Template  :\n{NUEXTRACT_TEMPLATE}")


# ==============================================================================
# SECTION 2 — HELPERS
# ==============================================================================
def build_nuextract_prompt(text: str) -> str:
    return (
        "<|input|>\n"
        f"{text.strip()}\n"
        "<|schema|>\n"
        f"{NUEXTRACT_TEMPLATE}\n"
        "<|output|>\n"
    )


def safe_int(val, default: int = 0) -> int:
    try:
        return int(str(val).strip())
    except (ValueError, TypeError):
        return default


def parse_nuextract_output(raw: str) -> dict:
    for stop in ["</s>", "<|endoftext|>", "<|end|>"]:
        raw = raw.split(stop)[0]
    raw = raw.strip()

    try:
        obj = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        try:
            obj = json.loads(m.group()) if m else {}
        except json.JSONDecodeError:
            obj = {}

    return {
        "Buildings_Fully_Inside":     safe_int(obj.get("Buildings_Fully_Inside",     0)),
        "Buildings_Partially_Inside": safe_int(obj.get("Buildings_Partially_Inside", 0)),
        "Total_Buildings_Impacted":   safe_int(obj.get("Total_Buildings_Impacted",   0)),
    }


def median_int(values: list) -> int:
    return int(round(statistics.median(values))) if values else 0


# ==============================================================================
# SECTION 3 — LOAD NuExtract WITH vLLM
# ==============================================================================
print(f"\n[INFO] Downloading {NUEXTRACT_MODEL_ID}...")
snapshot_download(repo_id=NUEXTRACT_MODEL_ID, resume_download=True, max_workers=8)
print("[INFO] Download complete.")

print(f"[INFO] Loading vLLM engine...")
nu_llm = LLM(
    model=NUEXTRACT_MODEL_ID,
    max_model_len=16384*2,
    trust_remote_code=True,
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    disable_log_stats=True,
)
print("✅ NuExtract vLLM engine ready")

nu_sampling = SamplingParams(
    temperature=0.0,
    max_tokens=256,
    stop=["</s>", "<|endoftext|>", "<|end|>"],
)


# ==============================================================================
# SECTION 4 — PROCESS CSVs
# ==============================================================================
raw_csvs = sorted(
    f for f in os.listdir(DRIVE_BASE)
    if f.startswith("vlm_raw_outputs_") and f.endswith(".csv")
)
print(f"\n[INFO] Found {len(raw_csvs)} raw CSV(s): {raw_csvs}")

for csv_file in raw_csvs:
    csv_path = os.path.join(DRIVE_BASE, csv_file)
    df       = pd.read_csv(csv_path)

    print(f"\n{'='*60}")
    print(f"📄 Extracting: {csv_file}  ({len(df)} rows)")
    print(f"{'='*60}")

    sample_cols = sorted(
        [c for c in df.columns if re.match(r"output_sample_\d+$", c)],
        key=lambda c: int(c.split("_")[-1]),
    )

    prompts, row_sample_map = [], []

    for df_idx, row in df.iterrows():
        for col in sample_cols:
            text = str(row.get(col, "")).strip()
            if not text or text.lower() == "nan":
                text = "No output available."
            prompts.append(build_nuextract_prompt(text))
            row_sample_map.append((df_idx, col))

    print(f"   Total extraction prompts: {len(prompts)}")

    print("[INFO] Running NuExtract inference...")
    nu_outputs = nu_llm.generate(prompts, nu_sampling, use_tqdm=True)
    print(f"✅ Extraction done — {len(nu_outputs)} outputs")

    per_row = {}
    for (df_idx, col), out in zip(row_sample_map, nu_outputs):
        raw_text = out.outputs[0].text if out.outputs else ""
        parsed   = parse_nuextract_output(raw_text)
        per_row.setdefault(df_idx, []).append(parsed)

    consensus_rows = []

    for df_idx, row in df.iterrows():
        samples = per_row.get(df_idx, [])
        base    = row.to_dict()

        if samples:
            fully_vals   = [s["Buildings_Fully_Inside"]     for s in samples]
            partial_vals = [s["Buildings_Partially_Inside"] for s in samples]
            total_vals   = [s["Total_Buildings_Impacted"]   for s in samples]

            fully   = median_int(fully_vals)
            partial = median_int(partial_vals)
            total   = median_int(total_vals)

            fully_std   = round(statistics.pstdev(fully_vals),   4)
            partial_std = round(statistics.pstdev(partial_vals), 4)
            total_std   = round(statistics.pstdev(total_vals),   4)

            fully_range   = max(fully_vals)   - min(fully_vals)
            partial_range = max(partial_vals) - min(partial_vals)
            total_range   = max(total_vals)   - min(total_vals)

            flat_samples = {}
            for k, s in enumerate(samples):
                flat_samples[f"s{k+1}_fully"]   = s["Buildings_Fully_Inside"]
                flat_samples[f"s{k+1}_partial"] = s["Buildings_Partially_Inside"]
                flat_samples[f"s{k+1}_total"]   = s["Total_Buildings_Impacted"]

        else:
            fully = partial = total = 0
            fully_std = partial_std = total_std = 0.0
            fully_range = partial_range = total_range = 0
            flat_samples = {}

        base.update({
            "extracted_fully_inside":     fully,
            "extracted_partially_inside": partial,
            "extracted_total_impacted":   total,

            "std_fully_inside":           fully_std,
            "std_partially_inside":       partial_std,
            "std_total_impacted":         total_std,
            "range_fully_inside":         fully_range,
            "range_partially_inside":     partial_range,
            "range_total_impacted":       total_range,

            **flat_samples,

            **{
                f"nuextract_sample_{k+1}": json.dumps(s, ensure_ascii=False)
                for k, s in enumerate(samples)
            },
        })

        consensus_rows.append(base)

    out_name = csv_file.replace("vlm_raw_outputs_", "nuextract_parsed_")
    out_path = os.path.join(DRIVE_BASE, out_name)
    pd.DataFrame(consensus_rows).to_csv(out_path, index=False)
    print(f"✅ Saved → {out_path}")


# ==============================================================================
# SECTION 5 — CLEANUP
# ==============================================================================
print("\n[INFO] Unloading NuExtract...")
del nu_llm
destroy_model_parallel()

if torch.distributed.is_initialized():
    torch.distributed.destroy_process_group()

gc.collect()
torch.cuda.empty_cache()

hf_folder  = f"models--{NUEXTRACT_MODEL_ID.replace('/', '--')}"
cache_path = os.path.join(HF_HOME, "hub", hf_folder)

if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print(f"✅ Disk cache deleted: {cache_path}")
else:
    print(f"⚠️  Cache not found, skipped: {cache_path}")

print("\n🎉 NuExtract extraction complete for all CSVs!")

In [ ]:
from google.colab import runtime
runtime.unassign()


# Eval

In [ ]:
import os
import glob
import json
import pandas as pd

# ==============================================================================
# 1. PATHS
# ==============================================================================
DIR_LLM_GEOSAM = "/content/drive/MyDrive/Outputs_3"
GROUND_TRUTH_CSV = "https://docs.google.com/spreadsheets/d/15tz3nDgU30rwycn3Itu6rT9r5WpaiNqgFM5vs6MACls/export?format=csv&gid=89410197"
BATCH_RESULTS_PATH = "/content/drive/MyDrive/Iran Israel War/crater_analysis/batch_results.json"

print("📥 Loading Data...")

# ==============================================================================
# 2. LOAD GROUND TRUTH & GEOSAM BASELINES
# ==============================================================================
# Load GT
gt_df = pd.read_csv(GROUND_TRUTH_CSV).reset_index(drop=True)
gt_df["gt_total"] = pd.to_numeric(gt_df["total_count"], errors="coerce")

# Load GeoSAM JSON
with open(BATCH_RESULTS_PATH, 'r') as f:
    batch_df = pd.DataFrame(json.load(f))

# Merge and calculate baseline errors
geo_merged = pd.merge(batch_df, gt_df, left_on='index', right_on='ID', how='inner')
geo_merged['google_total'] = geo_merged['google_complete'] + geo_merged['google_partial']
geo_merged['esri_total'] = geo_merged['esri_complete'] + geo_merged['esri_partial']

geo_merged['G_MAE'] = (geo_merged['google_total'] - geo_merged['gt_total']).abs()
geo_merged['E_MAE'] = (geo_merged['esri_total'] - geo_merged['gt_total']).abs()

# ==============================================================================
# 3. LOAD GEMMA (LLM + GEOSAM) DATA
# ==============================================================================
# Find the specific Gemma file in Outputs_3
csv_files = glob.glob(os.path.join(DIR_LLM_GEOSAM, "nuextract_parsed_google_gemma_4_31B_it.csv"))
if not csv_files:
    raise FileNotFoundError("Could not find the Gemma CSV file in Outputs_3")

gemma_file = csv_files[0]
df_gemma = pd.read_csv(gemma_file)

# Align with GT to extract the ID and calculate error
merged_gemma = pd.concat([df_gemma.reset_index(drop=True), gt_df.reset_index(drop=True)], axis=1)
preds = pd.to_numeric(merged_gemma['extracted_total_impacted'], errors='coerce').fillna(0)

gemma_results = pd.DataFrame({
    'ID': merged_gemma['ID'],  # Grab the ID from the GT side of the concat
    'GT_Count': merged_gemma['gt_total'],
    'Gemma_Pred': preds,
    'Gemma_MAE': (preds - merged_gemma['gt_total']).abs()
})

# ==============================================================================
# 4. COMPARE AND EXTRACT WINNING INDEXES
# ==============================================================================
# Combine Gemma results with baseline errors using the ID we just grabbed
comparison_df = pd.merge(
    gemma_results,
    geo_merged[['ID', 'google_total', 'esri_total', 'G_MAE', 'E_MAE']],
    on='ID',
    how='inner'
)

# Filter for instances where Gemma beat Google
beat_google = comparison_df[comparison_df['Gemma_MAE'] < comparison_df['G_MAE']].copy()
beat_google['margin'] = beat_google['G_MAE'] - beat_google['Gemma_MAE']
beat_google = beat_google.sort_values('margin', ascending=False)

# Filter for instances where Gemma beat ESRI
beat_esri = comparison_df[comparison_df['Gemma_MAE'] < comparison_df['E_MAE']].copy()
beat_esri['margin'] = beat_esri['E_MAE'] - beat_esri['Gemma_MAE']
beat_esri = beat_esri.sort_values('margin', ascending=False)

# ==============================================================================
# 5. OUTPUT
# ==============================================================================
print("\n" + "="*85)
print("🏆 INSTANCES WHERE GEMMA BEAT GOOGLE GEOSAM")
print("="*85)
print(f"Total Images: {len(beat_google)}\n")
cols_google = ['ID', 'GT_Count', 'google_total', 'Gemma_Pred', 'G_MAE', 'Gemma_MAE', 'margin']
print(beat_google[cols_google].to_string(index=False))

print("\n" + "="*85)
print("🏆 INSTANCES WHERE GEMMA BEAT ESRI GEOSAM")
print("="*85)
print(f"Total Images: {len(beat_esri)}\n")
cols_esri = ['ID', 'GT_Count', 'esri_total', 'Gemma_Pred', 'E_MAE', 'Gemma_MAE', 'margin']
print(beat_esri[cols_esri].to_string(index=False))

print("\n" + "="*85)
print("📋 RAW PYTHON LISTS (For easy copy/pasting)")
print("="*85)
print(f"beat_google_ids = {beat_google['ID'].tolist()}")
print(f"beat_esri_ids = {beat_esri['ID'].tolist()}")